In [1]:
import pandas as pd
import networkx as nx

df = pd.read_csv('export_api_influencers.csv')

# Limpieza Estructural
df = df.dropna(subset=['Cuentas_Que_Sigue'])
df['Usuario'] = df['Usuario'].str.strip().str.upper()
df = df.drop_duplicates(subset=['Usuario'])

# La Explosión
df['Cuentas_Que_Sigue'] = df['Cuentas_Que_Sigue'].str.split(',')
df = df.explode('Cuentas_Que_Sigue')

# Purificación Post-Explosión
df['Cuentas_Que_Sigue'] = df['Cuentas_Que_Sigue'].str.strip().str.upper()
df = df.rename(columns={'Usuario': 'Origen', 'Cuentas_Que_Sigue': 'Destino'})
df = df[df['Origen'] != df['Destino']][['Origen', 'Destino']]

display(df.head())

,Origen,Destino
0,BOT_5,AMATEUR_1
1,FAN_6,GARNACHO
1,FAN_6,MESSI
2,BOT_11,AMATEUR_1
3,BOT_1,AMATEUR_1


In [2]:
G = nx.from_pandas_edgelist(df, source='Origen', target='Destino', create_using=nx.DiGraph())

in_degree = nx.in_degree_centrality(G)
top_3_in = sorted(in_degree.items(), key=lambda x: x[1], reverse=True)[:3]

print("Top 3 Centralidad de Entrada:")
for user, score in top_3_in:
    print(f"{user}: {score:.4f}")

print("\n¿Quién parece ser el rey de la red a simple vista?")
print(f"R: {top_3_in[0][0]}, con el mayor porcentaje de conexiones entrantes ({top_3_in[0][1]*100:.2f}% de la red).")

Top 3 Centralidad de Entrada:
AMATEUR_1: 0.7231
GARNACHO: 0.2000
MESSI: 0.1538

¿Quién parece ser el rey de la red a simple vista?
R: AMATEUR_1, con el mayor porcentaje de conexiones entrantes (72.31% de la red).


In [3]:
pr = nx.pagerank(G)
top_3_pr = sorted(pr.items(), key=lambda x: x[1], reverse=True)[:3]

print("Top 3 PageRank:")
for user, score in top_3_pr:
    print(f"{user}: {score:.4f}")

G_undir = G.to_undirected()
clustering = nx.clustering(G_undir)

print(f"\nCoeficiente de Clustering AMATEUR_1: {clustering['AMATEUR_1']}")
print(f"Coeficiente de Clustering GARNACHO: {clustering['GARNACHO']}")

Top 3 PageRank:
AMATEUR_1: 0.1969
DE_PAUL: 0.0883
GARNACHO: 0.0843

Coeficiente de Clustering AMATEUR_1: 0
Coeficiente de Clustering GARNACHO: 0.13333333333333333


## Veredicto Final

### 1. ¿Quién compró bots y por qué?
**AMATEUR_1.**
- **Clustering = 0.0:** Sus 47 seguidores no están conectados entre sí; no forman comunidad ni triángulos, evidenciando una topología de estrella artificial característica de granjas de bots.
- **PageRank vs In-Degree:** Aunque concentra el 72.31% del in-degree por volumen bruto, su peso relativo cae en PageRank (0.1969) al recibir enlaces exclusivamente de nodos hoja sin autoridad.

### 2. ¿Quién es el influencer real y por qué?
**GARNACHO.**
- **Clustering = 0.1333:** Forma parte de una comunidad orgánica interconectada con múltiples triángulos cerrados entre seguidores y colegas.
- **Autoridad en PageRank:** Su métrica (0.0843) proviene de nodos de alto prestigio dentro del grafo (Messi, Suárez, Neymar, De Paul), validando su influencia real y orgánica en la red.